# Train the permissive QR detector (`@homstera/qr-detect`)

Trains a single-class QR detector (**Faster R-CNN + MobileNetV3-Large-FPN**, torchvision/BSD-3 — no Ultralytics/AGPL) on **synthetic data**, then exports `qr-detector.onnx`.

**How to use:** Runtime → *Change runtime type* → **GPU (T4)**, then **Runtime → Run all**. At the end, download `qr-detector.onnx` and drop it into the package at `models/qr-detector.onnx`. The Node package auto-detects it and uses it instead of the AGPL model.

Wall-clock on a free T4: dataset gen ~5–10 min, training ~1–2 h for 12 epochs on 8k images.

## 1. Setup — clone the repo and install non-torch deps
(torch/torchvision are preinstalled on Colab and version-matched; don't reinstall them for training.)

In [ ]:
!git clone https://github.com/princ3od/qr-detect.git || (cd qr-detect && git pull)
%cd qr-detect/training
import torch, torchvision
print('torch', torch.__version__, '| torchvision', torchvision.__version__,
      '| cuda', torch.cuda.is_available())

## 2. Generate the synthetic dataset
Tune `--train` up for better accuracy. 8k is a good starting point.

In [ ]:
!python generate_dataset.py --out data --train 8000 --val 500 --debug-grid
from IPython.display import Image as IPyImage
IPyImage('data/debug_grid.jpg')  # eyeball the labels: green boxes ONLY on QRs

## 3. Train
Watch `fp/img` (false positives per image) — it should fall toward **0**; `recall` should stay near **1.0**. `recall` alone is misleading. Let all 12 epochs finish so the LR fully anneals (suppresses false positives).

In [ ]:
!python train.py --data data --out runs --epochs 12 --batch-size 8 --lr 0.02 --workers 2

## 4. Export to ONNX  ⚠️ pinned torch
torchvision detection → ONNX is **version-fragile**: some torch builds emit a broken `roi_heads` Reshape that fails at inference (`Input shape:{N,7}, requested shape:{-1,4}`). We export with the **validated stack (torch 2.8.0 / torchvision 0.23.0, CPU)** — proven to run across input sizes in `onnxruntime`. No retrain needed; this only reloads the checkpoint and re-exports.

*(This replaces the GPU torch with the CPU build in this runtime. That's fine — training is already done. If you want to retrain afterwards, switch the runtime/reinstall GPU torch.)*

In [ ]:
!pip -q install torch==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cpu
!pip -q install onnx onnxruntime onnxscript
!python export_onnx.py --checkpoint runs/qr_frcnn_best.pth --out qr-detector.onnx

## 5. (Optional) Sanity-check on YOUR real cards
Upload a few real photos; this runs the exported ONNX with the **exact same preprocessing the Node package uses** (RGB /255, capped to 1024px max side) and draws detections ≥0.5. Expect a single tight box on the QR, no boxes on face/text/emblem.

In [ ]:
from google.colab import files
import numpy as np, onnxruntime as ort
from PIL import Image, ImageDraw, ImageOps

uploaded = files.upload()  # pick 1+ real card photos
sess = ort.InferenceSession('qr-detector.onnx', providers=['CPUExecutionProvider'])

def preprocess(path, max_side=1024):
    img = ImageOps.exif_transpose(Image.open(path).convert('RGB'))
    w, h = img.size
    s = min(1.0, max_side / max(w, h))
    if s < 1.0:
        img = img.resize((round(w*s), round(h*s)), Image.BICUBIC)
    arr = np.asarray(img, np.float32).transpose(2, 0, 1) / 255.0  # CHW RGB [0,1]
    return img, arr, s

for name in uploaded:
    disp, arr, s = preprocess(name)
    boxes, labels, scores = sess.run(None, {'images': arr})
    d = ImageDraw.Draw(disp)
    n = 0
    for b, sc in zip(boxes, scores):
        if sc < 0.5:
            continue
        n += 1
        d.rectangle(list(b), outline=(0, 255, 0), width=4)
        d.text((b[0], max(0, b[1]-12)), f'{sc:.2f}', fill=(0,255,0))
    print(name, '->', n, 'detections; top score',
          float(scores[0]) if len(scores) else None)
    display(disp)

## 6. Download the model
Drop the file into the package at `models/qr-detector.onnx`, then run `npm run build && npm test` locally.

In [ ]:
from google.colab import files
files.download('qr-detector.onnx')